# Module 02: Pandas for Machine Learning
## Notebook 02: Indexing, Filtering, and Safe Assignment

Subsetting data, selecting specific feature spaces, isolating anomalous rows, and avoiding subtle assignment bugs are daily machine learning tasks.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Distinguish between label-based (`.loc`) and integer position-based (`.iloc`) indexing.
2. Filter rows using compound boolean logic (`&`, `|`, `~`).
3. Leverage specialized selection methods (`.isin()`, `.between()`, `.query()`).
4. Prevent and diagnose the infamous `SettingWithCopyWarning`.
5. Safely assign derived features and conditional column transformations.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. Label-Based (`.loc`) vs. Position-Based (`.iloc`) Selection

- `.iloc[row_idx, col_idx]`: Integer position (0-indexed, behaves like standard Python slicing, half-open `[start:stop)`).
- `.loc[row_label, col_label]`: Label-based indexing (inclusive of both start and stop labels `[start:stop]`).

In [2]:
df = pd.DataFrame({
    'Feature_A': [10, 20, 30, 40, 50],
    'Feature_B': [1.5, 2.5, 3.5, 4.5, 5.5],
    'Category': ['Low', 'Low', 'Med', 'High', 'High']
}, index=['samp_1', 'samp_2', 'samp_3', 'samp_4', 'samp_5'])

print("DataFrame:\n", df)

# 1. Integer position with iloc (first 3 rows, first 2 columns)
print("\n--- .iloc[:3, :2] ---")
print(df.iloc[:3, :2])

# 2. Label based with loc ('samp_2' to 'samp_4', specific column names)
print("\n--- .loc['samp_2':'samp_4', ['Feature_A', 'Category']] ---")
print(df.loc['samp_2':'samp_4', ['Feature_A', 'Category']])

DataFrame:
         Feature_A  Feature_B Category
samp_1         10        1.5      Low
samp_2         20        2.5      Low
samp_3         30        3.5      Med
samp_4         40        4.5     High
samp_5         50        5.5     High

--- .iloc[:3, :2] ---
        Feature_A  Feature_B
samp_1         10        1.5
samp_2         20        2.5
samp_3         30        3.5

--- .loc['samp_2':'samp_4', ['Feature_A', 'Category']] ---
        Feature_A Category
samp_2         20      Low
samp_3         30      Med
samp_4         40     High


---
### 2. Boolean Filtering: Single & Compound Conditions

In Pandas, use:
- `&` for AND (both true)
- `|` for OR (at least one true)
- `~` for NOT (negation)
- **Always wrap each sub-clause in parentheses `()`** due to Python operator precedence!

In [3]:
# Filtering samples where Feature_A > 20 AND Category != 'High'
filter_mask = (df['Feature_A'] > 20) & (df['Category'] != 'High')
print("Boolean Mask:\n", filter_mask)
print("\nFiltered DataFrame:\n", df[filter_mask])

# Negation: Exclude 'Low' categories
print("\nExcluding 'Low' Category (~):\n", df[~(df['Category'] == 'Low')])

Boolean Mask:
 samp_1    False
samp_2    False
samp_3     True
samp_4    False
samp_5    False
dtype: bool

Filtered DataFrame:
         Feature_A  Feature_B Category
samp_3         30        3.5      Med

Excluding 'Low' Category (~):
         Feature_A  Feature_B Category
samp_3         30        3.5      Med
samp_4         40        4.5     High
samp_5         50        5.5     High


---
### 3. High-Level Filtering: `.isin()`, `.between()`, and `.query()`

Pandas provides clean, expressive methods that simplify complex conditional filtering.

In [4]:
# 1. .isin() - checking membership against a list of target categories
print("=== .isin(['Med', 'High']) ===")
print(df[df['Category'].isin(['Med', 'High'])])

# 2. .between() - closed numerical range [a, b]
print("\n=== .between(20, 40) on Feature_A ===")
print(df[df['Feature_A'].between(20, 40)])

# 3. .query() - SQL-like query string
threshold = 25
print("\n=== .query('Feature_A > @threshold and Category == \"Med\"') ===")
print(df.query("Feature_A > @threshold and Category == 'Med'"))

=== .isin(['Med', 'High']) ===
        Feature_A  Feature_B Category
samp_3         30        3.5      Med
samp_4         40        4.5     High
samp_5         50        5.5     High

=== .between(20, 40) on Feature_A ===
        Feature_A  Feature_B Category
samp_2         20        2.5      Low
samp_3         30        3.5      Med
samp_4         40        4.5     High

=== .query('Feature_A > @threshold and Category == "Med"') ===
        Feature_A  Feature_B Category
samp_3         30        3.5      Med


---
### 4. Avoiding the `SettingWithCopyWarning`

> **CRITICAL ML WARNING:**
> When you slice a DataFrame and attempt to assign a new column or modify values (chained indexing: `df[condition]['col'] = val`), Pandas cannot guarantee whether you are modifying a view or a copy, triggering a `SettingWithCopyWarning`.
> **The Fix:**
> - To modify the original: `df.loc[condition, 'col'] = val`
> - To create an independent sub-dataset: `subset = df[condition].copy()`

In [5]:
# Unsafe chained indexing (DO NOT DO THIS):
# df[df['Feature_A'] > 30]['Category'] = 'Outlier' -> SettingWithCopyWarning!

# Safe approach 1: Using .loc for in-place assignment
df_copy = df.copy()
df_copy.loc[df_copy['Feature_A'] > 30, 'Category'] = 'Verified_High'
print("Safely updated via .loc:\n", df_copy)

# Safe approach 2: Explicit .copy() when creating sub-datasets
high_subset = df[df['Feature_A'] >= 40].copy()
high_subset['Processed_Flag'] = True
print("\nIndependent subset with new column:\n", high_subset)

Safely updated via .loc:
         Feature_A  Feature_B       Category
samp_1         10        1.5            Low
samp_2         20        2.5            Low
samp_3         30        3.5            Med
samp_4         40        4.5  Verified_High
samp_5         50        5.5  Verified_High

Independent subset with new column:
         Feature_A  Feature_B Category  Processed_Flag
samp_4         40        4.5     High            True
samp_5         50        5.5     High            True


---
### 5. Creating Derived Features

Feature engineering often involves creating new columns based on existing numerical features or conditional logic.

In [6]:
# 1. Arithmetic feature combination (Interaction term)
df['Interaction_A_B'] = df['Feature_A'] * df['Feature_B']

# 2. Conditional feature using np.where
df['Is_Priority'] = np.where(df['Feature_A'] >= 30, 1, 0)

# 3. Non-linear transformation (log transformation for skewed features)
df['Log_Feature_A'] = np.log(df['Feature_A'])

print("DataFrame with newly engineered features:\n", df)

DataFrame with newly engineered features:
         Feature_A  Feature_B Category  Interaction_A_B  Is_Priority  \
samp_1         10        1.5      Low             15.0            0   
samp_2         20        2.5      Low             50.0            0   
samp_3         30        3.5      Med            105.0            1   
samp_4         40        4.5     High            180.0            1   
samp_5         50        5.5     High            275.0            1   

        Log_Feature_A  
samp_1       2.302585  
samp_2       2.995732  
samp_3       3.401197  
samp_4       3.688879  
samp_5       3.912023  


### Summary & Next Steps
In this notebook, you mastered:
- The distinction between `.loc` and `.iloc`.
- Complex multi-clause boolean filtering and `.query()`.
- Diagnosing and eliminating `SettingWithCopyWarning`.
- Vectorized derived feature creation.

**Next Notebook:** `03_data_cleaning_and_missing_values.ipynb` — Detect missing values, apply statistical imputation strategies, remove duplicates, and clean messy strings.